# Experiment 8 — Semantic Search using Embeddings and Vector Databases
**Aim:** Embed a document collection, index it with FAISS, and retrieve semantically related chunks.


# Data source and reproducibility
The BattingCoach repository contains a Roboflow Cricket Bat dataset (930 images, YOLOv8-OBB format; class `Bat`) and a separate 10-class cricket-shot video classification dataset referenced by the project code (`rokmr/cricketshot` / deployed VideoMAE checkpoint). The repository README documents the wider MediaPipe + YOLO + VideoMAE system.

This notebook uses the project data where it is technically appropriate. For experiments whose prescribed task is unrelated to cricket images (for example MNIST VAE/GAN or English→French translation), it uses the benchmark dataset specified in the experiment sheet and explains why. Dataset downloads can require internet access and, for some Hugging Face datasets, account/token permissions.

**Repository:** https://github.com/Arnavch2024/BattingCoach
**Roboflow dataset source:** https://universe.roboflow.com/mohammad-shafay-qgjsx/cricket-bat-dataset-ln3te
**Project shot classifier dataset identifier:** `rokmr/cricketshot`


### Dataset mapping
The most directly relevant structured data in BattingCoach is its project documentation and cricket-shot taxonomy. The repository README, dataset notes, and code descriptions can be represented as small text documents. This notebook uses a local text corpus describing the actual project and 10 shot labels, then demonstrates FAISS retrieval. It does not use the annotated image pixels as language documents.

## 1. Install

In [8]:
from importlib.metadata import version

for package in ('sentence-transformers', 'faiss-cpu'):
    try:
        print(f"{package}: {version(package)}")
    except Exception as exc:
        print(f"{package}: unavailable ({exc})")


sentence-transformers: 6.2.0
faiss-cpu: 1.15.1


## 2. Prepare project-domain documents

In [9]:
import pandas as pd
documents = [
    {"id":"D1","title":"System overview","source":"BattingCoach project documentation","text":"BatCoach AI Pro analyses cricket batting biomechanics using MediaPipe 3D pose landmarks and kinematic angles."},
    {"id":"D2","title":"Bat detection","source":"BattingCoach project documentation","text":"The BattingCoach project uses YOLOv8 Oriented Bounding Boxes to detect cricket bats and estimate blade angle."},
    {"id":"D3","title":"Bat dataset","source":"Roboflow Cricket Bat Dataset README","text":"The cricket bat detection dataset contains 930 images annotated in YOLOv8-OBB format with one class named Bat."},
    {"id":"D4","title":"Shot classifier","source":"BattingCoach project documentation","text":"The VideoMAE classifier recognizes ten cricket shots: cover, defense, flick, hook, late cut, lofted, pull, square cut, straight and sweep."},
    {"id":"D5","title":"Shadow practice","source":"BattingCoach project documentation","text":"Shadow practice mode suppresses bat-specific YOLO checks while retaining body-pose analysis."},
    {"id":"D6","title":"Live willow practice","source":"BattingCoach project documentation","text":"Live willow practice mode activates oriented bat detection and checks bat-pad gap and blade alignment."},
    {"id":"D7","title":"Biomechanics","source":"BattingCoach project documentation","text":"The biomechanics engine evaluates lead elbow elevation, knee flexion, head-over-knee balance and spine lean."},
    {"id":"D8","title":"Backend telemetry","source":"BattingCoach project documentation","text":"The project backend uses FastAPI and WebSockets for real-time coaching and session telemetry."},
    {"id":"D9","title":"Video classification","source":"BattingCoach project documentation","text":"A model may classify cover drive, pull, sweep and other stroke types from sampled video frames using VideoMAE."},
    {"id":"D10","title":"Dataset license","source":"Roboflow Cricket Bat Dataset README","text":"The Roboflow dataset README identifies the Cricket Bat Dataset and records its CC BY 4.0 license."}
]
texts=[d["text"] for d in documents]
metadata_table = pd.DataFrame([{k:d[k] for k in ("id","title","source")} for d in documents])
metadata_table


,id,title,source
0,D1,System overview,BattingCoach project documentation
1,D2,Bat detection,BattingCoach project documentation
2,D3,Bat dataset,Roboflow Cricket Bat Dataset README
3,D4,Shot classifier,BattingCoach project documentation
4,D5,Shadow practice,BattingCoach project documentation
5,D6,Live willow practice,BattingCoach project documentation
6,D7,Biomechanics,BattingCoach project documentation
7,D8,Backend telemetry,BattingCoach project documentation
8,D9,Video classification,BattingCoach project documentation
9,D10,Dataset license,Roboflow Cricket Bat Dataset README


## 3. Generate sentence embeddings


In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np
embed_model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors=embed_model.encode(texts,convert_to_numpy=True,normalize_embeddings=True)
print("Embedding model: sentence-transformers/all-MiniLM-L6-v2")
print("Embedding matrix shape:",doc_vectors.shape)
assert doc_vectors.shape == (len(documents), 384)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 575.54it/s]


Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding matrix shape: (10, 384)


## 4. Create a FAISS index


In [11]:
import faiss
vectors=doc_vectors.astype("float32")
index=faiss.IndexFlatIP(vectors.shape[1]) # cosine similarity because vectors are normalized
index.add(vectors)
print("FAISS index created; vectors:",index.ntotal,"dimension:",index.d)

FAISS index created; vectors: 10 dimension: 384


## 5. Implement semantic retrieval


In [12]:
def semantic_search(query,k=3):
    q=embed_model.encode([query],convert_to_numpy=True,normalize_embeddings=True).astype("float32")
    scores,ids=index.search(q,k)
    rows=[]
    for rank,(score,idx) in enumerate(zip(scores[0],ids[0]),start=1):
        d=documents[idx]
        rows.append({"rank":rank,"id":d["id"],"title":d["title"],"source":d["source"],"similarity":float(score),"text":d["text"]})
    return rows
queries=[
    ("How does the system identify the angle of a cricket bat?","close-source"),
    ("Which body posture values are checked for a batter?","close-source"),
    ("How does shadow batting work without a physical bat?","paraphrase"),
    ("What body mechanics does BatCoach measure?","paraphrase"),
    ("Tell me about marine biology and ocean currents.","absent")]
for q,kind in queries:
    print("\nQUERY:",q,"[",kind,"]")
    for row in semantic_search(q,3): print(row)



QUERY: How does the system identify the angle of a cricket bat? [ close-source ]
{'rank': 1, 'id': 'D2', 'title': 'Bat detection', 'source': 'BattingCoach project documentation', 'similarity': 0.7231473326683044, 'text': 'The BattingCoach project uses YOLOv8 Oriented Bounding Boxes to detect cricket bats and estimate blade angle.'}
{'rank': 2, 'id': 'D1', 'title': 'System overview', 'source': 'BattingCoach project documentation', 'similarity': 0.6159184575080872, 'text': 'BatCoach AI Pro analyses cricket batting biomechanics using MediaPipe 3D pose landmarks and kinematic angles.'}
{'rank': 3, 'id': 'D3', 'title': 'Bat dataset', 'source': 'Roboflow Cricket Bat Dataset README', 'similarity': 0.5820484161376953, 'text': 'The cricket bat detection dataset contains 930 images annotated in YOLOv8-OBB format with one class named Bat.'}

QUERY: Which body posture values are checked for a batter? [ close-source ]
{'rank': 1, 'id': 'D5', 'title': 'Shadow practice', 'source': 'BattingCoach proj

## 6. Evaluate paraphrase and irrelevant-query behaviour


In [13]:
test_queries=[
    ("How is bat orientation detected?","paraphrase",{"D2","D6"}),
    ("What body mechanics does BatCoach measure?","paraphrase",{"D1","D7"}),
    ("What does the application do when the batter is shadow practising?","paraphrase",{"D5"}),
    ("Which technology detects the bat?","close-source",{"D2"}),
    ("Explain photosynthesis in plants.","out-of-domain",set())]
evaluation=[]
for q,kind,expected_ids in test_queries:
    hits=semantic_search(q,3)
    for hit in hits:
        if kind == "out-of-domain":
            relevance = "Irrelevant" if hit["similarity"] < 0.20 else "Partially Relevant"
            reason = "The query is outside the cricket corpus; low similarity indicates no direct answer." if relevance == "Irrelevant" else "Weak overlap despite being outside the domain."
        elif hit["id"] in expected_ids and hit["rank"] == 1:
            relevance, reason = "Relevant", "The top result directly addresses the query intent."
        elif hit["id"] in expected_ids:
            relevance, reason = "Partially Relevant", "The result supports part of the query but is not the best match."
        else:
            relevance, reason = "Irrelevant", "The chunk does not support the requested fact."
        evaluation.append({"query":q,"type":kind,**hit,"relevance":relevance,"reason":reason})
import pandas as pd
evaluation_table=pd.DataFrame(evaluation)
evaluation_table[["query","type","rank","id","title","source","similarity","text","relevance","reason"]]


,query,type,rank,id,title,source,similarity,text,relevance,reason
0,How is bat orientation detected?,paraphrase,1,D6,Live willow practice,BattingCoach project documentation,0.634214,Live willow practice mode activates oriented b...,Relevant,The top result directly addresses the query in...
1,How is bat orientation detected?,paraphrase,2,D2,Bat detection,BattingCoach project documentation,0.590362,The BattingCoach project uses YOLOv8 Oriented ...,Partially Relevant,The result supports part of the query but is n...
2,How is bat orientation detected?,paraphrase,3,D5,Shadow practice,BattingCoach project documentation,0.515952,Shadow practice mode suppresses bat-specific Y...,Irrelevant,The chunk does not support the requested fact.
3,What body mechanics does BatCoach measure?,paraphrase,1,D1,System overview,BattingCoach project documentation,0.512855,BatCoach AI Pro analyses cricket batting biome...,Relevant,The top result directly addresses the query in...
4,What body mechanics does BatCoach measure?,paraphrase,2,D7,Biomechanics,BattingCoach project documentation,0.451302,The biomechanics engine evaluates lead elbow e...,Partially Relevant,The result supports part of the query but is n...
5,What body mechanics does BatCoach measure?,paraphrase,3,D10,Dataset license,Roboflow Cricket Bat Dataset README,0.407308,The Roboflow dataset README identifies the Cri...,Irrelevant,The chunk does not support the requested fact.
6,What does the application do when the batter i...,paraphrase,1,D5,Shadow practice,BattingCoach project documentation,0.577738,Shadow practice mode suppresses bat-specific Y...,Relevant,The top result directly addresses the query in...
7,What does the application do when the batter i...,paraphrase,2,D6,Live willow practice,BattingCoach project documentation,0.407549,Live willow practice mode activates oriented b...,Irrelevant,The chunk does not support the requested fact.
8,What does the application do when the batter i...,paraphrase,3,D2,Bat detection,BattingCoach project documentation,0.374422,The BattingCoach project uses YOLOv8 Oriented ...,Irrelevant,The chunk does not support the requested fact.
9,Which technology detects the bat?,close-source,1,D3,Bat dataset,Roboflow Cricket Bat Dataset README,0.573135,The cricket bat detection dataset contains 930...,Irrelevant,The chunk does not support the requested fact.


## 7. Compare with keyword matching

In [14]:
def keyword_rank(query):
    q=set(query.lower().split())
    scored=[]
    for d in documents:
        words=set(d["text"].lower().split())
        scored.append((len(q & words),d["id"],d["text"]))
    return sorted(scored,reverse=True)[:3]
query="How does the system identify the angle of a cricket bat?"
print("Semantic top-3:",[(x["id"],round(x["similarity"],3)) for x in semantic_search(query)])
print("Keyword top-3:",[(x[1],x[0]) for x in keyword_rank(query)])

Semantic top-3: [('D2', 0.723), ('D1', 0.616), ('D3', 0.582)]
Keyword top-3: [('D4', 2), ('D3', 2), ('D2', 2)]


## 8. Conclusion
The notebook reports the embedding model and dimension, FAISS inner-product metric on normalized vectors, document metadata, five test queries, ranked text chunks, similarity values, and reproducible relevance judgements. Similarity is a ranking score, not a calibrated probability. Semantic retrieval handles paraphrases better than exact keyword matching, while the absent-domain query demonstrates why a production system should apply a confidence threshold and abstain when no result is sufficiently similar.
